# 🚀 Vibe Training Notebook

**Training Vision Transformers for Interface Beauty Evaluation**

This notebook demonstrates how to train Vibe models using the WebDesignPrototypicality dataset with Google Colab's GPU acceleration.


## 🛠️ Setup and Preparation

In [ ]:
# Install and setup Vibe
!git clone https://github.com/glibas/Vibe.git
%cd Vibe
!pip install -q -r requirements.txt
!pip install -q -e .

# Quick environment setup
from vibe.data.colab_utils import quick_setup
quick_setup()

## 📊 Dataset Loading and Preparation

In [ ]:
from vibe.data.colab_utils import mount_drive_and_setup, load_webdesign_dataset, create_train_val_split

# Mount drive and load dataset
dataset_path = mount_drive_and_setup(
    '/content/drive/MyDrive/datasets/webdesignprototypicality.zip'
)

# Setup dataset structure
from vibe.data.colab_utils import setup_webdesign_dataset
data_paths = setup_webdesign_dataset(dataset_path)

# Create train/val/test splits
if data_paths['annotations_file']:
    split_files = create_train_val_split(
        data_paths['annotations_file'],
        train_ratio=0.7,
        val_ratio=0.2, 
        test_ratio=0.1
    )
    print("✅ Dataset splits created successfully")
else:
    print("⚠️ No annotations found, creating sample data")
    from vibe.data import create_sample_dataset
    create_sample_dataset('/content/sample_data', n_samples=200)

## 🧠 Model Configuration and Training

In [ ]:
import torch
from vibe.models import BeautyPredictor
from vibe.training import BeautyTrainer, BeautyLoss, create_optimizer, create_scheduler
from vibe.data import InterfaceDataLoader

# Model configuration
model_config = {
    'model_type': 'saliency_guided',  # Using saliency guidance
    'img_size': 224,
    'embed_dim': 512,  # Smaller for Colab
    'n_layers': 8,
    'n_heads': 8,
    'use_beauty_tokens': True,
    'saliency_weight': 0.5
}

# Create model
model = BeautyPredictor(**model_config)
print(f"🎯 Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

# Training configuration
training_config = {
    'num_epochs': 20,
    'batch_size': 16,  # Adjusted for Colab GPU
    'learning_rate': 1e-4,
    'weight_decay': 0.01
}

print("📋 Training configuration:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

In [ ]:
# Create data loaders
if 'split_files' in locals():
    # Use real dataset splits
    train_loader = InterfaceDataLoader.create_train_loader(
        data_path=data_paths['dataset_root'],
        annotations_file=split_files['train'],
        batch_size=training_config['batch_size'],
        augmentation=True
    )
    
    val_loader = InterfaceDataLoader.create_val_loader(
        data_path=data_paths['dataset_root'],
        annotations_file=split_files['val'],
        batch_size=training_config['batch_size']
    )
else:
    # Use sample data
    train_loader = InterfaceDataLoader.create_train_loader(
        data_path='/content/sample_data',
        annotations_file='/content/sample_data/annotations.csv',
        batch_size=training_config['batch_size']
    )
    val_loader = train_loader  # Use same for demo

print(f"📊 Training samples: {len(train_loader.dataset)}")
print(f"📊 Validation samples: {len(val_loader.dataset)}")

In [ ]:
# Setup training components
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🔧 Using device: {device}")

# Optimizer and scheduler
optimizer = create_optimizer(model, {
    'type': 'adamw',
    'lr': training_config['learning_rate'],
    'weight_decay': training_config['weight_decay']
})

scheduler = create_scheduler(optimizer, {
    'type': 'cosine',
    'T_max': training_config['num_epochs']
})

# Loss function
loss_fn = BeautyLoss(
    beauty_weight=1.0,
    aspect_weight=0.5,
    principle_weight=0.3,
    consistency_weight=0.2
)

print("✅ Training components configured")

## 🚂 Model Training

In [ ]:
# Create trainer
trainer = BeautyTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_fn=loss_fn,
    device=device,
    log_dir='/content/logs',
    save_dir='/content/checkpoints',
    eval_interval=2,
    save_interval=5
)

print("🚂 Starting training...")
print("=" * 50)

# Start training
trainer.train(training_config['num_epochs'])

print("\n✅ Training completed!")
print(f"📁 Model saved to: /content/checkpoints/best.pth")

## 📊 Training Results and Evaluation

In [ ]:
# Load best model for evaluation
best_model_path = '/content/checkpoints/best.pth'
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Loaded best model from epoch {checkpoint['epoch']}")
    
    # Quick evaluation
    model.eval()
    test_sample = next(iter(val_loader))
    
    with torch.no_grad():
        images = test_sample['image'][:1].to(device)
        analysis = model.analyze_interface(images)
        explanation = model.get_beauty_explanation(analysis)
        
        print("\n🎨 Sample Analysis:")
        print("=" * 40)
        print(explanation)
        
    print(f"\n📈 Training history available in: /content/checkpoints/training_history.json")
else:
    print("⚠️ No trained model found")

## 💾 Save Results to Google Drive

In [ ]:
# Save trained model and results to Google Drive
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_save_dir = f'/content/drive/MyDrive/vibe_training_{timestamp}'

# Create directory in Drive
os.makedirs(drive_save_dir, exist_ok=True)

# Copy important files
files_to_save = [
    ('/content/checkpoints/best.pth', 'best_model.pth'),
    ('/content/checkpoints/training_history.json', 'training_history.json'),
    ('/content/logs', 'logs')
]

for src, dst in files_to_save:
    dst_path = os.path.join(drive_save_dir, dst)
    if os.path.exists(src):
        if os.path.isdir(src):
            shutil.copytree(src, dst_path, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst_path)
        print(f"✅ Saved: {dst}")

print(f"\n💾 Training results saved to: {drive_save_dir}")
print("\n🎯 Next steps:")
print("- Use EvaluationNotebook.ipynb for detailed evaluation")
print("- Use AnalysisNotebook.ipynb for interface analysis")
print("- Access saved model for inference in future sessions")